# Pandas Labs – Limpieza, análisis y estructuración de datos
**GA1-220501094-01-AA1-EV06** · Proyecto: *Análisis de la siniestralidad vial en Colombia*

Este cuaderno continúa el diagnóstico de calidad de la EV05 (`data_quality.ipynb`) y desarrolla en orden:

| Lección | Tema | Laboratorio guiado | Reto |
|---|---|---|---|
| 2 | Limpieza y estandarización | Pipeline de limpieza con informe de cambios | Reto 2 (7 ejercicios) |
| 3 | Filtrado, ordenamiento y agrupación | Perfil de la siniestralidad vial (temporal y territorial) | Reto 3 (7 ejercicios) |
| 4 | Combinación de datasets | Tabla de análisis por municipio | Reto 4 (7 ejercicios) |

**Pregunta orientadora (adaptada al proyecto):** ¿Cómo preparar y estructurar los datos de siniestralidad vial para obtener información confiable sobre dónde y cuándo se concentran los lesionados y fallecidos?

> Equivalencias con el proyecto guía Instacart: *órdenes → registros de lesionados*, *productos → municipios*, *pasillos/departamentos → departamentos y sectores viales*, *hábitos de compra → patrones de siniestralidad*.

In [1]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

BASE = Path("..") if Path("../data").exists() else Path(".")
RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Pandas", pd.__version__)
sorted(p.name for p in RAW.glob("*.csv"))

Pandas 3.0.5


['lesiones_accidentes_transito.csv',
 'sectores_criticos_siniestralidad_vial.csv']

---
# Lección 2. Limpieza y estandarización
## 2.1 Ejemplos de la lección
Se cargan los datos **originales** (como en la EV05) para aplicar sobre ellos cada técnica.

In [2]:
lesiones_raw = pd.read_csv(RAW / "lesiones_accidentes_transito.csv")
sectores_raw = pd.read_csv(RAW / "sectores_criticos_siniestralidad_vial.csv")
print(lesiones_raw.shape, sectores_raw.shape)
lesiones_raw.dtypes

(644042, 6) (316, 13)


FECHA HECHO       str
COD_DEPTO       int64
DEPARTAMENTO      str
COD_MUNI        int64
MUNICIPIO         str
CANTIDAD        int64
dtype: object

### Ejemplo 1 – Renombrar columnas a `snake_case`

In [3]:
def a_snake_case(columnas):
    return (pd.Index(columnas).str.strip().str.lower()
            .str.replace(" ", "_", regex=False))

ej = sectores_raw.copy()
ej.columns = a_snake_case(ej.columns)
ej = ej.rename(columns={"gizscore": "gi_zscore", "gipvalue": "gi_pvalue"})
list(ej.columns)

['id_mt',
 'entidad',
 'gi_zscore',
 'fallecidos',
 'gi_pvalue',
 'tramo',
 'nombre',
 'latitud',
 'longitud',
 'pr',
 'municipio',
 'departamento',
 'divipola']

### Ejemplo 2 – Estandarizar texto: espacios, mayúsculas y tildes
Se quitan tildes pero se conserva la **Ñ** (NARIÑO no debe convertirse en NARINO).

In [4]:
TABLA_TILDES = str.maketrans("ÁÉÍÓÚÜáéíóúü", "AEIOUUaeiouu")

def normalizar_texto(serie):
    return (serie.astype("string")
                 .str.strip()
                 .str.upper()
                 .str.replace(r"\s+", " ", regex=True)
                 .str.translate(TABLA_TILDES))

pd.DataFrame({"original": sectores_raw["Departamento"].unique()[:8],
              "normalizado": normalizar_texto(pd.Series(sectores_raw["Departamento"].unique()[:8]))})

,original,normalizado
0,CUNDINAMARCA,CUNDINAMARCA
1,META,META
2,BOYACÁ,BOYACA
3,SANTANDER,SANTANDER
4,QUINDÍO,QUINDIO
5,TOLIMA,TOLIMA
6,CESAR,CESAR
7,RISARALDA,RISARALDA


### Ejemplo 3 – Convertir tipos: fechas, números con coma decimal y códigos con ceros

In [5]:
fecha = pd.to_datetime(lesiones_raw["FECHA HECHO"], format="%d/%m/%Y", errors="coerce")
zscore = pd.to_numeric(sectores_raw["GiZScore"].str.replace(",", ".", regex=False), errors="coerce")
cod_muni = lesiones_raw["COD_MUNI"].astype(str).str.zfill(5)

print("Fecha  :", lesiones_raw["FECHA HECHO"].dtype, "->", fecha.dtype, "| NaT:", fecha.isna().sum())
print("ZScore :", sectores_raw["GiZScore"].dtype, "->", zscore.dtype, "| NaN:", zscore.isna().sum())
print("Código :", lesiones_raw["COD_MUNI"].iloc[85], "->", cod_muni.iloc[85])

Fecha  : str -> datetime64[us] | NaT: 0
ZScore : str -> float64 | NaN: 0
Código : 5045 -> 05045


### Ejemplo 4 – Valores faltantes ocultos → `NaN`, y tratamiento con `fillna` / `dropna`

In [6]:
ej = sectores_raw[["ID_MT", "PR", "Nombre"]].replace(["<Null>", "sd"], np.nan)
print("Nulos después de reemplazar marcadores:"); print(ej.isna().sum())
print("\nfillna('SIN NOMBRE') ->", ej["Nombre"].fillna("SIN NOMBRE").isna().sum(), "nulos")
print("dropna(subset=['PR']) ->", len(ej.dropna(subset=["PR"])), "de", len(ej), "filas")

Nulos después de reemplazar marcadores:
ID_MT     175
PR         42
Nombre    175
dtype: int64

fillna('SIN NOMBRE') -> 0 nulos
dropna(subset=['PR']) -> 274 de 316 filas


### Ejemplo 5 – Duplicados: completos y por subconjunto de columnas

In [7]:
print("Filas idénticas (lesiones):", lesiones_raw.duplicated().sum())
print("Coordenadas repetidas (sectores):", sectores_raw.duplicated(subset=["Latitud", "Longitud"]).sum())
print("drop_duplicates por coordenadas:", len(sectores_raw), "->",
      len(sectores_raw.drop_duplicates(subset=["Latitud", "Longitud"])))

Filas idénticas (lesiones): 294354
Coordenadas repetidas (sectores): 4
drop_duplicates por coordenadas: 316 -> 312


### Ejemplo 6 – Valores atípicos con rango intercuartílico (IQR)

In [8]:
f = pd.to_numeric(sectores_raw["Fallecidos"])
q1, q3 = f.quantile([0.25, 0.75])
iqr = q3 - q1
lim_sup = q3 + 1.5 * iqr
print(f"Q1={q1}, Q3={q3}, IQR={iqr}, límite superior={lim_sup}")
print("Sectores con fallecidos atípicos:", (f > lim_sup).sum())

Q1=7.0, Q3=14.0, IQR=7.0, límite superior=24.5
Sectores con fallecidos atípicos: 15


## 2.2 Laboratorio guiado 2 – Pipeline de limpieza con informe de cambios
Cada paso es una función que recibe un DataFrame, devuelve uno nuevo y **registra** lo que cambió (filas antes/después, valores modificados y descripción). Al final se obtiene un informe de cambios auditable.

In [9]:
informe_cambios = []

def registrar(dataset, paso, df_antes, df_despues, valores_afectados, descripcion):
    informe_cambios.append({
        "dataset": dataset, "paso": paso,
        "filas_antes": len(df_antes), "filas_despues": len(df_despues),
        "valores_afectados": int(valores_afectados), "descripcion": descripcion,
    })
    return df_despues

MARCADORES_NULOS = ["SIN ESTABLECER", "<Null>", "sd"]
NOMBRE_MES = {1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril", 5: "Mayo", 6: "Junio", 7: "Julio",
              8: "Agosto", 9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"}
NOMBRE_DIA = {0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves", 4: "Viernes", 5: "Sábado", 6: "Domingo"}

### Pipeline para `lesiones`

In [10]:
def limpiar_lesiones(raw):
    d = "lesiones"
    # 1. Renombrar columnas
    df = raw.rename(columns={"FECHA HECHO": "fecha", "COD_DEPTO": "cod_depto", "DEPARTAMENTO": "departamento",
                             "COD_MUNI": "cod_muni", "MUNICIPIO": "municipio", "CANTIDAD": "cantidad"})
    df = registrar(d, "1. Renombrar columnas", raw, df, raw.shape[1], "Nombres a snake_case en español")

    # 2. Fecha a datetime
    antes = df; df = df.copy()
    df["fecha"] = pd.to_datetime(df["fecha"], format="%d/%m/%Y", errors="coerce")
    df = registrar(d, "2. Convertir fecha", antes, df, len(df), f"texto -> datetime64; NaT: {df['fecha'].isna().sum()}")

    # 3. Códigos DIVIPOLA como texto con ceros
    antes = df; df = df.copy()
    afectados = (df["cod_muni"] < 10000).sum()
    df["cod_depto"] = df["cod_depto"].astype(str).str.zfill(2)
    df["cod_muni"] = df["cod_muni"].astype(str).str.zfill(5)
    df = registrar(d, "3. Códigos DIVIPOLA", antes, df, afectados, "int -> texto con zfill(2)/zfill(5)")

    # 4. Normalizar texto
    antes = df; df = df.copy()
    for c in ["departamento", "municipio"]:
        df[c] = normalizar_texto(df[c])
    cambios = (antes[["departamento", "municipio"]].astype("string") != df[["departamento", "municipio"]]).sum().sum()
    df = registrar(d, "4. Normalizar texto", antes, df, cambios, "strip, mayúsculas, espacios y tildes")

    # 5. Nulos ocultos 'SIN ESTABLECER' -> NaN (también su código)
    antes = df; df = df.copy()
    sin_depto = df["departamento"].isin(MARCADORES_NULOS)
    sin_muni = df["municipio"].isin(MARCADORES_NULOS)
    df.loc[sin_depto, ["departamento", "cod_depto"]] = pd.NA
    df.loc[sin_muni, ["municipio", "cod_muni"]] = pd.NA
    df = registrar(d, "5. Nulos ocultos", antes, df, sin_depto.sum() + sin_muni.sum(), "'SIN ESTABLECER' -> NaN")

    # 6. Variables derivadas de la fecha
    antes = df; df = df.copy()
    df["anio"] = df["fecha"].dt.year
    df["mes"] = df["fecha"].dt.month
    df["nombre_mes"] = df["mes"].map(NOMBRE_MES)
    df["dia_semana"] = df["fecha"].dt.dayofweek.map(NOMBRE_DIA)
    df["anio_parcial"] = df["anio"] == df["fecha"].max().year   # 2026 llega hasta septiembre
    df = registrar(d, "6. Variables de fecha", antes, df, 5, "anio, mes, nombre_mes, dia_semana, anio_parcial")

    # 7. Marcar atípicos (no se eliminan: son eventos reales con muchos lesionados)
    antes = df; df = df.copy()
    p999 = df["cantidad"].quantile(0.999)
    df["cantidad_atipica"] = df["cantidad"] > p999
    df = registrar(d, "7. Marcar atípicos", antes, df, df["cantidad_atipica"].sum(), f"cantidad > percentil 99,9 ({p999:.0f})")

    # 8. Duplicados: se conservan (no hay ID de víctima), solo se documentan
    df = registrar(d, "8. Duplicados", df, df, df.duplicated().sum(), "Se conservan: sin ID de víctima no son errores")

    # 9. Tipos categóricos para ahorrar memoria
    antes = df; df = df.copy()
    for c in ["departamento", "municipio", "dia_semana", "nombre_mes"]:
        df[c] = df[c].astype("category")
    df = registrar(d, "9. Tipos categóricos", antes, df, 4, "texto repetitivo -> category")
    return df

lesiones = limpiar_lesiones(lesiones_raw)
lesiones.head()

,fecha,cod_depto,departamento,cod_muni,municipio,cantidad,anio,mes,nombre_mes,dia_semana,anio_parcial,cantidad_atipica
0,2026-09-30,76,VALLE DEL CAUCA,76001,CALI,1,2026,9,Septiembre,Miércoles,True,False
1,2026-09-30,76,VALLE DEL CAUCA,76001,CALI,1,2026,9,Septiembre,Miércoles,True,False
2,2026-09-30,76,VALLE DEL CAUCA,76001,CALI,1,2026,9,Septiembre,Miércoles,True,False
3,2026-09-30,76,VALLE DEL CAUCA,76001,CALI,1,2026,9,Septiembre,Miércoles,True,False
4,2026-09-30,70,SUCRE,70001,SINCELEJO,1,2026,9,Septiembre,Miércoles,True,False


### Pipeline para `sectores`

In [11]:
def limpiar_sectores(raw):
    d = "sectores"
    # 1. Renombrar columnas
    df = raw.copy()
    df.columns = a_snake_case(df.columns)
    df = df.rename(columns={"gizscore": "gi_zscore", "gipvalue": "gi_pvalue", "divipola": "cod_muni"})
    df = registrar(d, "1. Renombrar columnas", raw, df, raw.shape[1], "snake_case; divipola -> cod_muni")

    # 2. Nulos ocultos
    antes = df
    afectados = df.isin(MARCADORES_NULOS).sum().sum()
    df = df.replace(MARCADORES_NULOS, np.nan)
    df = registrar(d, "2. Nulos ocultos", antes, df, afectados, "'<Null>' y 'sd' -> NaN")

    # 3. Coma decimal -> float y PR numérico
    antes = df; df = df.copy()
    for c in ["gi_zscore", "gi_pvalue", "latitud", "longitud"]:
        df[c] = pd.to_numeric(df[c].str.replace(",", ".", regex=False))
    df["pr"] = pd.to_numeric(df["pr"])
    df = registrar(d, "3. Tipos numéricos", antes, df, 5 * len(df), "gi_zscore, gi_pvalue, latitud, longitud, pr -> float")

    # 4. Código DIVIPOLA
    antes = df; df = df.copy()
    afectados = (df["cod_muni"] < 10000).sum()
    df["cod_muni"] = df["cod_muni"].astype(str).str.zfill(5)
    df["cod_depto"] = df["cod_muni"].str[:2]
    df = registrar(d, "4. Código DIVIPOLA", antes, df, afectados, "cod_muni zfill(5) y nuevo cod_depto")

    # 5. Normalizar texto
    antes = df; df = df.copy()
    for c in ["departamento", "municipio", "tramo", "nombre", "entidad"]:
        df[c] = normalizar_texto(df[c])
    df["tramo"] = df["tramo"].str.replace("*", "", regex=False).str.strip()
    cambios = sum((antes[c].astype("string") != df[c]).fillna(False).sum() for c in ["departamento", "municipio", "tramo", "nombre"])
    df = registrar(d, "5. Normalizar texto", antes, df, cambios, "tildes, mayúsculas, espacios dobles, '*' en tramo")

    # 6. Imputar descriptivos faltantes
    antes = df; df = df.copy()
    afectados = df[["tramo", "nombre"]].isna().sum().sum()
    df["tramo"] = df["tramo"].fillna("SIN TRAMO")
    df["nombre"] = df["nombre"].fillna("SIN NOMBRE")
    df = registrar(d, "6. Imputar texto faltante", antes, df, afectados, "tramo/nombre nulos -> 'SIN TRAMO'/'SIN NOMBRE'")

    # 7. Duplicados por coordenadas: conservar el de mayor gi_zscore
    antes = df
    df = (df.sort_values("gi_zscore", ascending=False)
            .drop_duplicates(subset=["latitud", "longitud"], keep="first")
            .sort_index())
    df = registrar(d, "7. Duplicados por coordenadas", antes, df, len(antes) - len(df), "Se conserva el punto con mayor gi_zscore")

    # 8. Nivel de significancia del punto caliente
    antes = df; df = df.copy()
    df["nivel_confianza"] = pd.cut(df["gi_pvalue"], bins=[-0.001, 0.01, 0.05, 0.10],
                                   labels=["99%", "95%", "90%"])
    df = registrar(d, "8. Nivel de confianza", antes, df, len(df), "gi_pvalue -> 99% / 95% / 90%")

    # 9. Tipos categóricos
    antes = df; df = df.copy()
    df["entidad"] = df["entidad"].astype("category")
    df = registrar(d, "9. Tipos categóricos", antes, df, 1, "entidad -> category")
    return df.reset_index(drop=True)

sectores = limpiar_sectores(sectores_raw)
sectores.head()

,id_mt,entidad,gi_zscore,fallecidos,gi_pvalue,tramo,nombre,latitud,longitud,pr,municipio,departamento,cod_muni,cod_depto,nivel_confianza
0,4G013,ANI,1.77,11,0.08,BOGOTA - VILLAVICENCIO,IP – CHIRAJARA - FUNDADORES,4.45,-74.05,10.00,CHIPAQUE,CUNDINAMARCA,25178,25,90%
1,4G013,ANI,3.82,23,0.00,BOGOTA - VILLAVICENCIO,IP – CHIRAJARA - FUNDADORES,4.23,-73.82,58.00,GUAYABETAL,CUNDINAMARCA,25335,25,99%
2,4G013,ANI,2.80,1,0.01,BOGOTA - VILLAVICENCIO,IP – CHIRAJARA - FUNDADORES,4.20,-73.81,60.00,GUAYABETAL,CUNDINAMARCA,25335,25,99%
3,4G013,ANI,1.73,1,0.08,BOGOTA - VILLAVICENCIO,IP – CHIRAJARA - FUNDADORES,4.12,-73.64,85.00,VILLAVICENCIO,META,50001,50,90%
4,NaN,INVIAS-OTROS,1.73,13,0.08,GRANADA - VILLAVICENCIO,SIN NOMBRE,4.12,-73.65,NaN,VILLAVICENCIO,META,50001,50,90%


### Informe de cambios

In [12]:
reporte_limpieza = pd.DataFrame(informe_cambios)
reporte_limpieza

,dataset,paso,filas_antes,filas_despues,valores_afectados,descripcion
0,lesiones,1. Renombrar columnas,644042,644042,6,Nombres a snake_case en español
1,lesiones,2. Convertir fecha,644042,644042,644042,texto -> datetime64; NaT: 0
2,lesiones,3. Códigos DIVIPOLA,644042,644042,80832,int -> texto con zfill(2)/zfill(5)
3,lesiones,4. Normalizar texto,644042,644042,585,"strip, mayúsculas, espacios y tildes"
4,lesiones,5. Nulos ocultos,644042,644042,14,'SIN ESTABLECER' -> NaN
5,lesiones,6. Variables de fecha,644042,644042,5,"anio, mes, nombre_mes, dia_semana, anio_parcial"
6,lesiones,7. Marcar atípicos,644042,644042,638,"cantidad > percentil 99,9 (21)"
7,lesiones,8. Duplicados,644042,644042,294354,Se conservan: sin ID de víctima no son errores
8,lesiones,9. Tipos categóricos,644042,644042,4,texto repetitivo -> category
9,sectores,1. Renombrar columnas,316,316,13,snake_case; divipola -> cod_muni


In [13]:
print("Tipos finales – lesiones"); print(lesiones.dtypes, "\n")
print("Tipos finales – sectores"); print(sectores.dtypes)

Tipos finales – lesiones
fecha               datetime64[us]
cod_depto                      str
departamento              category
cod_muni                       str
municipio                 category
cantidad                     int64
anio                         int32
mes                          int32
nombre_mes                category
dia_semana                category
anio_parcial                  bool
cantidad_atipica              bool
dtype: object 

Tipos finales – sectores
id_mt                   str
entidad            category
gi_zscore           float64
fallecidos            int64
gi_pvalue           float64
tramo                string
nombre               string
latitud             float64
longitud            float64
pr                  float64
municipio            string
departamento         string
cod_muni                str
cod_depto               str
nivel_confianza    category
dtype: object


In [14]:
print("Nulos restantes – lesiones:"); print(lesiones.isna().sum()[lambda s: s > 0], "\n")
print("Nulos restantes – sectores:"); print(sectores.isna().sum()[lambda s: s > 0])

Nulos restantes – lesiones:
cod_depto        3
departamento     3
cod_muni        11
municipio       11
dtype: int64 

Nulos restantes – sectores:
id_mt    175
pr        42
dtype: int64


**Lectura del informe:** en `lesiones` no se elimina ninguna fila (644.042 antes y después). Los cambios son de **tipo y formato**, más 14 ubicaciones `SIN ESTABLECER` que pasan a nulo. En `sectores` se eliminan 4 puntos con coordenadas repetidas (316 → 312). Los únicos nulos que quedan son esperables: `id_mt` (vías sin contrato de concesión) y `pr` (sin punto de referencia).

## 2.3 Reto 2 – 7 ejercicios

**Ejercicio 2.1.** Compare los valores faltantes de `sectores` antes y después de la limpieza (incluidos los nulos ocultos).

In [15]:
antes_ocultos = sectores_raw.isna().sum() + sectores_raw.isin(MARCADORES_NULOS).sum()
antes_ocultos.index = a_snake_case(antes_ocultos.index)
antes_ocultos = antes_ocultos.rename({"gizscore": "gi_zscore", "gipvalue": "gi_pvalue", "divipola": "cod_muni"})
comparacion = pd.DataFrame({"antes": antes_ocultos, "despues": sectores.isna().sum()}).fillna(0).astype(int)
comparacion[(comparacion["antes"] > 0) | (comparacion["despues"] > 0)]

,antes,despues
id_mt,175,175
nombre,175,0
pr,42,42
tramo,23,0


**Ejercicio 2.2.** Verifique que, después de normalizar, todos los departamentos de `sectores` coinciden con los de `lesiones`.

In [16]:
dep_les = set(lesiones["departamento"].dropna().unique())
dep_sec_raw = set(sectores_raw["Departamento"].unique())
dep_sec = set(sectores["departamento"].unique())
print("Sin coincidencia ANTES  :", sorted(dep_sec_raw - set(lesiones_raw["DEPARTAMENTO"])))
print("Sin coincidencia DESPUÉS:", sorted(dep_sec - dep_les))

Sin coincidencia ANTES  : ['ATLÁNTICO', 'BOLÍVAR', 'BOYACÁ', 'CAQUETÁ', 'CÓRDOBA', 'QUINDÍO']
Sin coincidencia DESPUÉS: []


**Ejercicio 2.3.** Compruebe que las columnas de fecha derivadas son coherentes: sin `NaT`, meses de 1 a 12 y el número de días por año.

In [17]:
print("NaT en fecha:", lesiones["fecha"].isna().sum())
print("Meses:", sorted(lesiones["mes"].unique().tolist()))
print("Año parcial:", lesiones.loc[lesiones["anio_parcial"], "anio"].unique(),
      "| último mes:", lesiones.loc[lesiones["anio_parcial"], "mes"].max())
lesiones.groupby("anio")["fecha"].nunique().rename("dias_con_registro").to_frame().T

NaT en fecha: 0
Meses: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Año parcial: [2026] | último mes: 9


anio,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
dias_con_registro,365,366,365,365,365,366,365,365,365,366,365,365,365,366,365,365,365,366,365,365,365,366,365,273


**Ejercicio 2.4.** Compare dos criterios de atípicos para `cantidad`: IQR y percentil 99,9. Explique cuál conviene.

In [18]:
q1, q3 = lesiones["cantidad"].quantile([0.25, 0.75])
lim_iqr = q3 + 1.5 * (q3 - q1)
p999 = lesiones["cantidad"].quantile(0.999)
print(f"IQR: Q1={q1}, Q3={q3} -> límite {lim_iqr} -> atípicos: {(lesiones['cantidad'] > lim_iqr).sum():,}")
print(f"P99.9: límite {p999:.0f} -> atípicos: {(lesiones['cantidad'] > p999).sum():,}")
lesiones.loc[lesiones["cantidad_atipica"], ["fecha", "municipio", "cantidad"]].nlargest(5, "cantidad")

IQR: Q1=1.0, Q3=1.0 -> límite 1.0 -> atípicos: 79,726
P99.9: límite 21 -> atípicos: 638


,fecha,municipio,cantidad
422749,2016-04-25,BOGOTA D.C.,91
380444,2017-12-02,BOGOTA D.C.,63
421206,2016-05-16,BOGOTA D.C.,60
378776,2017-12-29,CALI,56
508731,2011-01-01,MEDELLIN,53


> Como el 75 % de los registros tiene `cantidad = 1`, el IQR es 0 y marcaría como atípico **cualquier** registro con más de un lesionado (≈12 %). El percentil 99,9 es más razonable: marca solo eventos excepcionales, que se conservan y se señalan.

**Ejercicio 2.5.** Cuantifique los duplicados y demuestre que eliminarlos alteraría el total de lesionados.

In [19]:
total_real = lesiones["cantidad"].sum()
sin_dup = lesiones.drop_duplicates()
print(f"Duplicados exactos: {lesiones.duplicated().sum():,}")
print(f"Total lesionados conservando filas: {total_real:,}")
print(f"Total si se eliminaran duplicados : {sin_dup['cantidad'].sum():,}  "
      f"(pérdida {1 - sin_dup['cantidad'].sum() / total_real:.1%})")

Duplicados exactos: 294,354
Total lesionados conservando filas: 885,829
Total si se eliminaran duplicados : 589,932  (pérdida 33.4%)


**Ejercicio 2.6.** Reemplace las siglas de `entidad` por nombres descriptivos con `map()`.

In [20]:
ENTIDADES = {"ANI": "Agencia Nacional de Infraestructura", "INVIAS-OTROS": "INVÍAS y otras vías nacionales",
             "DEPTO": "Vía departamental", "CIUDAD": "Vía urbana"}
sectores["entidad_nombre"] = sectores["entidad"].astype(str).map(ENTIDADES).astype("category")
print("Sin mapear:", sectores["entidad_nombre"].isna().sum())
sectores["entidad_nombre"].value_counts()

Sin mapear: 0


entidad_nombre
Agencia Nacional de Infraestructura    137
INVÍAS y otras vías nacionales         126
Vía departamental                       48
Vía urbana                               1
Name: count, dtype: int64

**Ejercicio 2.7.** Mida el ahorro de memoria obtenido con los tipos `category`.

In [21]:
mem_raw = lesiones_raw.memory_usage(deep=True).sum() / 1024**2
sin_cat = lesiones.astype({c: "string" for c in ["departamento", "municipio", "dia_semana", "nombre_mes"]})
mem_str = sin_cat.memory_usage(deep=True).sum() / 1024**2
mem_cat = lesiones.memory_usage(deep=True).sum() / 1024**2
pd.DataFrame({"MB": [mem_raw, mem_str, mem_cat]},
             index=["original (6 columnas)", "limpio con texto", "limpio con category"])

,MB
original (6 columnas),122.76
limpio con texto,224.73
limpio con category,83.59


### Guardar los datos limpios

In [22]:
lesiones.to_csv(PROCESSED / "lesiones_clean.csv", index=False)
sectores.to_csv(PROCESSED / "sectores_clean.csv", index=False)
reporte_limpieza.to_csv(PROCESSED / "cleaning_log.csv", index=False)
sorted(p.name for p in PROCESSED.glob("*.csv"))

['cleaning_log.csv', 'lesiones_clean.csv', 'sectores_clean.csv']